In [ ]:

import streamlit as st

from travel_backend import (
    load_data,
    register_user,
    user_login,
    add_record,
    view_records,
    search_record,
    generate_report,
    USERS_FILE,
    TRIPS_FILE
)

# Page settings
st.set_page_config(
    page_title="TravelMate",
    page_icon="✈️",
    layout="wide"
)

# Load saved data
users = load_data(USERS_FILE, {})
trips = load_data(TRIPS_FILE, [])

if not isinstance(users, dict):
    users = {}

if not isinstance(trips, list):
    trips = []

# Session state
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False

if "username" not in st.session_state:
    st.session_state.username = ""


# Website heading
st.title("✈️ TravelMate")
st.subheader("Your Personal Travel Planning Portal")
st.write("Plan your journeys, manage your budgets and explore destinations!")

st.divider()


# LOGIN AND REGISTRATION PAGE
if not st.session_state.logged_in:

    login_tab, register_tab = st.tabs(["Login", "Register"])

    # Login
    with login_tab:

        st.header("Welcome Back!")

        with st.form("login_form"):

            username = st.text_input("Username")
            password = st.text_input("Password", type="password")

            login_button = st.form_submit_button("Login")

            if login_button:

                if user_login(users, username, password):

                    st.session_state.logged_in = True
                    st.session_state.username = username.strip()

                    st.success("Login successful!")
                    st.rerun()

                else:
                    st.error("Invalid username or password.")


    # Registration
    with register_tab:

        st.header("Create Your Account")

        with st.form("register_form"):

            new_username = st.text_input("Choose a Username")
            new_password = st.text_input(
                "Choose a Password",
                type="password"
            )

            confirm_password = st.text_input(
                "Confirm Password",
                type="password"
            )

            register_button = st.form_submit_button("Register")

            if register_button:

                if new_password != confirm_password:

                    st.error("Passwords do not match.")

                else:

                    success = register_user(
                        users,
                        new_username,
                        new_password
                    )

                    if success:
                        st.success("Account created successfully! You can now log in.")
                    else:
                        st.error("Registration failed. Username may already exist or information may be invalid.")


# MAIN PORTAL AFTER LOGIN
else:

    username = st.session_state.username

    st.sidebar.title("TravelMate Menu")
    st.sidebar.write("Welcome, " + username + "!")

    choice = st.sidebar.radio(
        "Select a Section",
        [
            "Dashboard",
            "Add Travel Plan",
            "My Travel Plans",
            "Search Destination",
            "Travel Report"
        ]
    )

    st.sidebar.divider()

    if st.sidebar.button("Logout"):

        st.session_state.logged_in = False
        st.session_state.username = ""

        st.rerun()


    # DASHBOARD
    if choice == "Dashboard":

        st.header("Your Dashboard")

        my_trips = view_records(trips, username)

        total_trips = len(my_trips)
        total_budget = 0
        total_days = 0

        for trip in my_trips:
            total_budget += trip["budget"]
            total_days += trip["days"]

        col1, col2, col3 = st.columns(3)

        col1.metric("Total Trips", total_trips)
        col2.metric("Total Budget", "Rs. " + str(total_budget))
        col3.metric("Total Travel Days", total_days)

        st.subheader("Recent Travel Plans")

        if len(my_trips) == 0:

            st.info("You have not added any travel plans yet.")

        else:

            for trip in my_trips[-3:][::-1]:

                st.write("**Destination:**", trip["destination"])
                st.write("**Country:**", trip["country"])
                st.write("**Budget:** Rs.", trip["budget"])
                st.divider()


    # ADD TRAVEL PLAN
    elif choice == "Add Travel Plan":

        st.header("Add a New Travel Plan")

        with st.form("travel_form"):

            destination = st.text_input("Destination")
            country = st.text_input("Country")

            days = st.number_input(
                "Number of Days",
                min_value=1,
                step=1
            )

            budget = st.number_input(
                "Estimated Budget (Rs.)",
                min_value=0.01,
                step=1000.0
            )

            trip_type = st.selectbox(
                "Type of Travel",
                [
                    "Solo",
                    "Family",
                    "Adventure",
                    "Business",
                    "Couple"
                ]
            )

            notes = st.text_area("Additional Notes")

            add_button = st.form_submit_button("Save Travel Plan")

            if add_button:

                success = add_record(
                    trips,
                    username,
                    destination,
                    country,
                    days,
                    budget,
                    trip_type,
                    notes
                )

                if success:

                    st.success("Travel plan added successfully!")
                    st.rerun()

                else:

                    st.error("Unable to add travel plan. Please check your information.")


    # MY TRAVEL PLANS
    elif choice == "My Travel Plans":

        st.header("My Travel Plans")

        my_trips = view_records(trips, username)

        if len(my_trips) == 0:

            st.info("No travel plans available.")

        else:

            for number, trip in enumerate(my_trips, start=1):

                st.subheader("Travel Plan " + str(number))

                st.write("**Destination:**", trip["destination"])
                st.write("**Country:**", trip["country"])
                st.write("**Duration:**", trip["days"], "days")
                st.write("**Budget:** Rs.", trip["budget"])
                st.write("**Travel Type:**", trip["trip_type"])
                st.write("**Notes:**", trip["notes"])

                st.divider()


    # SEARCH DESTINATION
    elif choice == "Search Destination":

        st.header("Search Your Travel Plans")

        destination = st.text_input("Enter destination to search")

        if st.button("Search"):

            if destination.strip() == "":

                st.warning("Please enter a destination.")

            else:

                results = search_record(
                    trips,
                    username,
                    destination
                )

                if len(results) == 0:

                    st.warning("No matching travel plans found.")

                else:

                    st.success("Matching travel plans found!")

                    for trip in results:

                        st.write("**Destination:**", trip["destination"])
                        st.write("**Country:**", trip["country"])
                        st.write("**Duration:**", trip["days"], "days")
                        st.write("**Budget:** Rs.", trip["budget"])
                        st.divider()


    # TRAVEL REPORT
    elif choice == "Travel Report":

        st.header("Your Travel Report")

        report = generate_report(trips, username)

        if report is None:

            st.info("Add a travel plan to generate your report.")

        else:

            col1, col2, col3 = st.columns(3)

            col1.metric(
                "Total Trips",
                report["total_trips"]
            )

            col2.metric(
                "Total Budget",
                "Rs. " + str(report["total_budget"])
            )

            col3.metric(
                "Average Budget",
                "Rs. " + str(round(report["average_budget"], 2))
            )

            st.subheader("Travel Destinations")

            my_trips = view_records(trips, username)

            for trip in my_trips:

                st.write(
                    trip["destination"],
                    "-",
                    trip["country"]
                )
